# Federation Demo  
In this demo, you will be shown:  
- how to see the datasets imported
- how to search for data
- merge data from different tables
- add a table to an existing database

Some Imports

In [1]:
import csv
from getpass import getpass
from pathlib import Path

from dsi.dsi import DSI
from dsi.dsifederated import DSIFederated
from dsi.sync import Sync
from dsi.utils.federated.federate_datasets import pull_data

from dsi.utils.federation_utils import (
    compute_md5, 
    create_directory, 
    create_folder_from_path, 
    csv_to_list_of_dicts, 
    deduplicate_keep_latest, 
    get_last_part, 
    human_readable_size, 
    should_download, 
    upsert_records
)

Help for DSI and Federated DSI

In [2]:
help(DSI)

Help on class DSI in module dsi.dsi:

class DSI(builtins.object)
 |  DSI(filename='.temp_dsi.db', backend_name='Sqlite', **kwargs)
 |
 |  A user-facing interface for DSI's Core middleware.
 |
 |  The DSI Class abstracts Core.Terminal for managing metadata and Core.Sync for data management and movement.
 |
 |  Methods defined here:
 |
 |  __init__(self, filename='.temp_dsi.db', backend_name='Sqlite', **kwargs)
 |      Initializes DSI by activating a backend for data operations; default is a Sqlite backend for temporary data analysis.
 |      If users specify `filename`, data is saved to a permanent backend file.
 |
 |      `filename` : str, optional, default is ".temp_dsi.db"
 |          If not specified, a temporary, hidden backend file is created for users to analyze their data.
 |          If specified and backend file already exists, it is activated for a user to explore its data.
 |          If specified and backend file does not exist, a file with this name is created.
 |
 |      

In [3]:
help(DSIFederated)

Help on class DSIFederated in module dsi.dsifederated:

class DSIFederated(builtins.object)
 |  DSIFederated(federated_folder_path: str, operating_mode: str = 'console')
 |
 |  A class for federated querying of DSI databases. It loads metadata about the databases and
 |  their tables from a specified folder, and provides methods to summarize, query, search, and find data across the federated databases.
 |
 |  Methods defined here:
 |
 |  __init__(self, federated_folder_path: str, operating_mode: str = 'console')
 |      Initializes the DSIFederated class by loading metadata about the federated databases and their tables from a specified folder.
 |
 |      Args:
 |          federated_folder_path (str): The file path to the folder containing the metadata about the federated databases. The folder should contain a JSON file named "dsi_database_list.json" with the metadata information.
 |          operating_mode (str): console or notebook, determines how the results are displayed. Default i

In [4]:
help(Sync)

Help on class Sync in module dsi.sync:

class Sync(builtins.object)
 |  Sync(project_name, isVerbose=False, no_parent=False, skip_index=False, **kwargs)
 |
 |  A class defined to assist in data management activities for DSI
 |
 |  Sync is where data movement functions such as copy (to remote location) and
 |  sync (local filesystem with remote) exist.
 |
 |  Methods defined here:
 |
 |  __init__(self, project_name, isVerbose=False, no_parent=False, skip_index=False, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  change_group(self, local_loc, user_group)
 |      Change group permissions for data and db. Only works for OS with Unix (not Windows)
 |
 |  change_permissions(self, local_loc)
 |      Change read permissions for data and db. Only works for OS with Unix (not Windows)
 |
 |  copy(self, tool='copy')
 |      Helper function to perform the data copy over using a preferred API
 |
 |  dircrawl(self, filepath, verbose=False)
 |      Crawls the 

In [5]:
filename = "other_db.csv"

In [6]:
with open(filename, 'r') as file:
    csv_reader = csv.DictReader(file)
    csv_data = list(csv_reader)

In [7]:
#csv_data

In [8]:
rel_wrks_folder = "test_11"

In [9]:
workspace_folder = str(Path(rel_wrks_folder).resolve())

In [10]:
database_info = []
federation_dbs = []

In [11]:
success_counter = 0
for row in csv_data:
    username = ""
    password = ""
    if row['location_type'].strip().lower() == "hpc":
        print(f"\n{'='*60}")
        print(f"Enter credentials for data at { row["location"]} : {row["path"]}")
        username = input("Username: ")
        password = getpass("Password: ")  # Hidden input!

    db_info = pull_data(location_type=row['location_type'],
              location=row['location'],
              path=row['path'],
              abs_path_workspace_folder=workspace_folder,
              username=username,
              password=password,
              internal_use=False)
    
    if db_info:
        database_info.append(db_info)
        combined = {k: row[k] for k in ["location_type", "location", "submitter_name"]} | {k: db_info[k] for k in ["local_path", "name", "folder_hash"]}
        combined["workspace_folder"] = workspace_folder
        federation_dbs.append(combined)
        success_counter += 1
              

# # Save host_usernames to a file for future runs
# with open(f"{workspace_folder}/host_usernames.json", "w", encoding="utf-8") as f:
#         yaml.safe_dump(host_username, f)

# Save databases information to a JSON file
upsert_records(f"{workspace_folder}/dsi_database_list.json", database_info, key="original_path")


Enter credentials for data at ch-fe.lanl.gov : /lustre/scratch5/pascalgrosset/test_db/nif.db


Username:  pascalgrosset
Password:  ········




 - Processing database at HPC:ch-fe.lanl.gov:/lustre/scratch5/pascalgrosset/test_db/nif.db
hostname ch-fe.lanl.gov, username: pascalgrosset, len(password): 51, remote_path: /lustre/scratch5/pascalgrosset/test_db/nif.db
File size: 20480 bytes
Success!!!


 - Processing database at url:url:https://www.timestored.com/data/sample/sakila.db
Downloaded to /home/pascalgrosset/projects/dsi/examples/federated/test_10/03a0e743fce29ae0/sakila.db

Enter credentials for data at darwin-fe.lanl.gov : /users/pulido/modelcard2.db


Username:  pascalgrosset
Password:  ········




 - Processing database at HPC:darwin-fe.lanl.gov:/users/pulido/modelcard2.db
hostname darwin-fe.lanl.gov, username: pascalgrosset, len(password): 0, remote_path: /users/pulido/modelcard2.db
!!!! !!!! !!! Error: [Errno 2] No such file
 -- Could not access or download the file at darwin-fe.lanl.gov:/users/pulido/modelcard2.db. Skipping this database.


 - Processing database at github:github:https://github.com/lanl/dsi/tree/ai_dev/tools/ai_search/data/oceans_11/ocean_11_datasets.db


 - Processing database at S3:S3:s3://cloudtrail90161934391290952873/wildfire.db
 -- Permission error: Access denied for s3://cloudtrail90161934391290952873/wildfire.db. Skipping this database.


NameError: name 'abs_path_workspace_folder' is not defined

In [15]:
database_info

[{'original_location_type': 'ch-fe.lanl.gov',
  'original_path': '/lustre/scratch5/pascalgrosset/test_db/nif.db',
  'folder_hash': '78592646fc355251',
  'local_path': '/home/pascalgrosset/projects/dsi/examples/federated/test_10/78592646fc355251/nif.db',
  'name': 'nif.db'},
 {'original_location_type': 'url',
  'original_path': 'https://www.timestored.com/data/sample/sakila.db',
  'folder_hash': '03a0e743fce29ae0',
  'local_path': '/home/pascalgrosset/projects/dsi/examples/federated/test_10/03a0e743fce29ae0/sakila.db',
  'name': 'sakila.db'},
 {'original_location_type': 'github',
  'original_path': 'https://github.com/lanl/dsi/tree/ai_dev/tools/ai_search/data/oceans_11/ocean_11_datasets.db',
  'folder_hash': '17576f97cca3c0df',
  'local_path': '/home/pascalgrosset/projects/dsi/examples/federated/test_10/17576f97cca3c0df/ocean_11_datasets.db',
  'name': 'ocean_11_datasets.db'}]

In [12]:
# Save databases information to a JSON file
upsert_records(f"{workspace_folder}/dsi_database_list.json", database_info, key="original_path")

## Instantiate the object

In [13]:
federated_dbs = DSIFederated(workspace_folder, operating_mode="notebook")

## Browse and search through the data

In [14]:
federated_dbs.f_list_databases()

,id,original_location,original_path,name,path,num_tables,tables
0,bald-raptor,ch-fe.lanl.gov,/lustre/scratch5/pascalgrosset/test_db/nif.db,nif.db,/home/pascalgrosset/projects/dsi/examples/fede...,2,"[array_and_types, nif_metadata]"
1,nickel-horse,url,https://www.timestored.com/data/sample/sakila.db,sakila.db,/home/pascalgrosset/projects/dsi/examples/fede...,16,"[actor, country, city, address, language, cate..."
2,heretic-smilodon,github,https://github.com/lanl/dsi/tree/ai_dev/tools/...,ocean_11_datasets.db,/home/pascalgrosset/projects/dsi/examples/fede...,1,[genesis_datacard]


## Looking up data

In [16]:
federated_dbs.f_summary()


Database: nif.db at path /home/pascalgrosset/projects/dsi/examples/federated/test_10/78592646fc355251/nif.db:


[       column     type unique   min   max   avg std_dev
 0  array_name  VARCHAR      9  None  None  None    None
 1  array_type  VARCHAR      1  None  None  None    None,
         column     type unique            min                max  \
 0     sim_name  VARCHAR     20           None               None   
 1     timestep      INT     20           None               None   
 2   num_arrays      INT      1           None               None   
 3        shape  VARCHAR     19           None               None   
 4     ablt_min    FLOAT      1            0.0                0.0   
 5     ablt_max    FLOAT      1            1.0                1.0   
 6     cham_min    FLOAT      1            0.0                0.0   
 7     cham_max    FLOAT      1            1.0                1.0   
 8     dens_min    FLOAT     17       0.000007           0.000017   
 9     dens_max    FLOAT     20        2.63112           7.954777   
 10    depo_min    FLOAT      1            0.0                0.0   



Database: sakila.db at path /home/pascalgrosset/projects/dsi/examples/federated/test_10/03a0e743fce29ae0/sakila.db:


[        column         type unique   min   max    avg    std_dev
 0    actor_id*      NUMERIC    200     1   200  100.5  57.734305
 1   first_name  VARCHAR(45)    128  None  None   None       None
 2    last_name  VARCHAR(45)    121  None  None   None       None
 3  last_update    TIMESTAMP      3  None  None   None       None,
         column         type unique   min   max   avg std_dev
 0  country_id*     SMALLINT    109  None  None  None    None
 1      country  VARCHAR(50)    109  None  None  None    None
 2  last_update    TIMESTAMP      1  None  None  None    None,
         column         type unique   min   max   avg std_dev
 0     city_id*          INT    600  None  None  None    None
 1         city  VARCHAR(50)    599  None  None  None    None
 2   country_id     SMALLINT    109  None  None  None    None
 3  last_update    TIMESTAMP      6  None  None  None    None,
         column         type unique   min   max   avg std_dev
 0  address_id*          INT    603  None  None


Database: ocean_11_datasets.db at path /home/pascalgrosset/projects/dsi/examples/federated/test_10/17576f97cca3c0df/ocean_11_datasets.db:


[                                               column     type unique   min  \
 0                                               Title  VARCHAR      9  None   
 1                                        Keywords/Tag  VARCHAR      9  None   
 2                            Update/Modification_Date  VARCHAR      6  None   
 3                             Theme/Category_:_Domain  VARCHAR      5  None   
 4                                        Description_  VARCHAR      9  None   
 5                                      Contact_Point_  VARCHAR      6  None   
 6   Formatted_Name*Format_with_separate_fields_for...  VARCHAR      5  None   
 7                                               Email  VARCHAR      5  None   
 8   Access_Rights_:_SecurityClassification_(Inform...  VARCHAR      1  None   
 9                                     CUI_Restriction  VARCHAR      1  None   
 10                                 CUI_Banner_Marking  VARCHAR      1  None   
 11                           CUI_Design

In [17]:
federated_dbs.f_search(query="dens_max")

[[     table_name column_name
  0  nif_metadata    dens_max],
 None,
 None]

## Merging data

### Search for databases

In [ ]:
federated_dbs.f_search_for_databases(db="data_subset*")

### Search for the path to a database

In [ ]:
federated_dbs.f_get_db_path(db="data_subset_1.db")

### Load that database

In [ ]:
temp_file = DSI('/Users/pascalgrosset/projects/dsi/dsi_databases_00/87c1361f3c2316dd/data_subset_1.db')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='camouflaged-hare',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='precious-walrus',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

## Adding another table to the database

In [ ]:
federated_dbs.f_search_for_databases(db="model_subset*")

In [ ]:
federated_dbs.f_add_table(src_db_id='satisfied-whale',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='model')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("model", collection=True)

That database now has datasets which have been pulled from different sites as well as several tables 